# BG-forecasting — seed-major grid on Colab

Runs the publication grid with **F=4** inputs (interstitial glucose, basal insulin,
insulin bolus, carbohydrate intake), one seed at a time across the whole grid:

```
seed 41  ->  gru / lstm / rnn  x  15 / 30 / 45 / 60 min   (12 patients, both modes)
seed 42  ->  the same twelve cells
seed 43  ->  the same twelve cells
```

This notebook is a driver for two scripts in the repo — `run_seed_major.sh` (runs
the cells, skipping finished ones) and `merge_seed_runs.py` (recombines the three
seeds per cell). It adds what Colab needs: Drive staging, and a mirror to Drive
after **every** cell so a disconnect costs you at most one cell.

---

### Read before running

**The dataset is DUA-restricted.** OhioT1DM is governed by a data use agreement.
This notebook expects a copy in your Drive, which places it on third-party
storage. Check your agreement permits that before you upload anything. Nothing
here does it for you.

**Budget about 0.6 h per cell, ~22 h for all 36.** Colab free caps sessions near
12 h and disconnects after ~90 min idle, so you will not finish in one session.
That is fine — progress is mirrored to Drive after each cell and finished cells
are skipped on the next run.

**Leave `device: cpu` alone.** These are small recurrent models at batch 64 over
12 timesteps; a GPU buys little, and the configs pin CPU for bitwise
reproducibility.

## 1 · Mount Drive and set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# --- edit these to match your Drive ---------------------------------------
DRIVE_DATA    = "/content/drive/MyDrive/ohiot1dm"      # must contain 2018/ and 2020/
DRIVE_RESULTS = "/content/drive/MyDrive/bg-results"    # results are mirrored here
REPO_DIR      = "/content/BG-forecasting"

# Where the code comes from.
#   "git"   -> clone REPO_URL at BRANCH
#   "drive" -> copy DRIVE_REPO (use this if the branch is not pushed)
SOURCE     = "git"
REPO_URL   = "https://github.com/beatriz-fulgencio/BG-forecasting.git"
BRANCH     = "bench2"
DRIVE_REPO = "/content/drive/MyDrive/BG-forecasting"

SEEDS    = [41, 42, 43]
MODELS   = ["gru", "lstm", "rnn"]
HORIZONS = [15, 30, 45, 60]
# ---------------------------------------------------------------------------

import os, pathlib
for d in (DRIVE_RESULTS, f"{DRIVE_RESULTS}/experiments", f"{DRIVE_RESULTS}/logs"):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
print("Drive ready.")

## 2 · Get the code

`run_seed_major.sh`, `merge_seed_runs.py` and the F=4 `configs/full_*.yaml` must
be present. If they are not committed and pushed, set `SOURCE = "drive"` above and
put a copy of the repo folder in Drive instead — the check below tells you which
files are missing rather than failing later.

In [ ]:
import shutil, subprocess, sys, pathlib

if SOURCE == "git":
    if not pathlib.Path(REPO_DIR).is_dir():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
elif SOURCE == "drive":
    if not pathlib.Path(REPO_DIR).is_dir():
        shutil.copytree(DRIVE_REPO, REPO_DIR)
else:
    raise ValueError("SOURCE must be 'git' or 'drive'")

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

required = ["run_seed_major.sh", "merge_seed_runs.py"] + [
    f"configs/full_{m}_{h}min.yaml" for m in MODELS for h in HORIZONS
]
missing = [f for f in required if not pathlib.Path(f).is_file()]
if missing:
    raise SystemExit(
        "Missing from this checkout:\n  " + "\n  ".join(missing) +
        "\n\nCommit and push them, or set SOURCE = 'drive'."
    )
os.chmod("run_seed_major.sh", 0o755)
print(f"Repo ready at {REPO_DIR}; all {len(required)} required files present.")

## 3 · Stage the OhioT1DM data

In [ ]:
import shutil, pathlib

dst = pathlib.Path(REPO_DIR) / "data" / "raw" / "ohiot1dm"
src = pathlib.Path(DRIVE_DATA)
if not src.is_dir():
    raise SystemExit(f"{DRIVE_DATA} not found. Upload your OhioT1DM copy there first.")

for release in ("2018", "2020"):
    target = dst / release
    if target.is_dir():
        continue
    source = src / release
    if not source.is_dir():
        raise SystemExit(f"Expected {source} (with train/ and test/ inside).")
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, target)

# 6 patients x 2 releases x train+test = 24 XML files.
found = sorted(dst.glob("*/*/*.xml"))
print(f"{len(found)} XML file(s) staged under {dst}")
for release, expected in (("2018", [559,563,570,575,588,591]), ("2020", [540,544,552,567,584,596])):
    for mode in ("train", "test"):
        names = {p.name for p in (dst/release/mode).glob("*.xml")}
        missing = [p for p in expected if f"{p}-ws-{mode}ing.xml" not in names]
        print(f"  {release}/{mode}: {len(names)} file(s)" + (f"  MISSING {missing}" if missing else ""))
if len(found) != 24:
    print("\nWarning: expected 24 files. Runs will fail on whatever is absent.")

## 4 · Restore earlier progress, then check the plan

Pulls anything already finished back from Drive so finished cells are skipped,
then prints the plan without running anything.

In [ ]:
import pathlib, shutil, subprocess

EXP_DIR = pathlib.Path(REPO_DIR) / "results" / "experiments"
EXP_DIR.mkdir(parents=True, exist_ok=True)

restored = 0
for d in sorted(pathlib.Path(f"{DRIVE_RESULTS}/experiments").glob("experiment_*")):
    target = EXP_DIR / d.name
    if not target.exists():
        shutil.copytree(d, target)
        restored += 1
print(f"Restored {restored} experiment dir(s) from Drive.\n")

env = dict(os.environ, DRY_RUN="1",
           SEEDS=" ".join(map(str, SEEDS)),
           MODELS=" ".join(MODELS),
           HORIZONS=" ".join(map(str, HORIZONS)),
           PYTHON=sys.executable)
print(subprocess.run(["./run_seed_major.sh"], cwd=REPO_DIR, env=env,
                     capture_output=True, text=True).stdout)

## 5 · Confirm the inputs really are F=4

Builds one patient's dataset — preprocessing only, no training — and asserts the
four feature columns. Cheap insurance against discovering at analysis time that
the runs were glucose-only.

In [ ]:
from benchmark.data.loaders import load_ohiot1dm_data
from benchmark.data.preprocessors import preprocess_ohiot1dm_data
from benchmark.data.torch_dataset import prepare_patient_datasets
import io, contextlib

pid = 559
with contextlib.redirect_stdout(io.StringIO()):          # preprocessing is chatty
    tr = preprocess_ohiot1dm_data(
        load_ohiot1dm_data("data", patient_ids=[pid], mode="train", version="2018", sampling_rate=5),
        include_feature_engineering=False)
    te = preprocess_ohiot1dm_data(
        load_ohiot1dm_data("data", patient_ids=[pid], mode="test", version="2018", sampling_rate=5),
        include_feature_engineering=False)
    train_ds, test_ds = prepare_patient_datasets(tr[pid], te[pid], 12, 3, unimodal=False)

print("features:", train_ds.feature_columns)
print("train:", train_ds.data.shape, " test:", test_ds.data.shape)
assert train_ds.feature_columns == ["glucose", "basal", "bolus", "carbs"], train_ds.feature_columns
assert train_ds.data.shape[1] == 4
print("\nF=4 confirmed.")

## 6 · Run

One cell at a time, seed-major, mirroring to Drive after each. Re-running this
cell after a disconnect picks up where it stopped. Full logs go to Drive;
only progress lines are printed here.

In [ ]:
import subprocess, shutil, pathlib, time, sys, os

def mirror_to_drive():
    """Copy completed experiment dirs to Drive. Incomplete runs are skipped."""
    dst = pathlib.Path(f"{DRIVE_RESULTS}/experiments")
    copied = 0
    for d in EXP_DIR.glob("experiment_*"):
        if not (d / "aggregate_metrics.json").is_file():
            continue
        target = dst / d.name
        if target.exists():
            continue
        shutil.copytree(d, target)
        copied += 1
    return copied

KEEP = ("[", "seed ", "  ok ", "  FAILED", "Preflight", "Nothing", "Done:", "config:", "ERROR", "Traceback")

def run_cell(seed, model, horizon):
    name = f"full_{model}_{horizon}min_seed{seed}"
    env = dict(os.environ, SEEDS=str(seed), MODELS=model, HORIZONS=str(horizon),
               PYTHON=sys.executable, LOG_DIR=f"{DRIVE_RESULTS}/logs")
    t0 = time.time()
    proc = subprocess.Popen(["./run_seed_major.sh"], cwd=REPO_DIR, env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        if any(k in line for k in KEEP):
            print(line.rstrip(), flush=True)
    rc = proc.wait()
    copied = mirror_to_drive()
    print(f"    {'OK ' if rc == 0 else 'FAIL'} {name} in {(time.time()-t0)/60:.0f} min"
          f"  (mirrored {copied} dir(s) to Drive)", flush=True)
    return rc

failures = []
for seed in SEEDS:                      # seed-major: the outer loop is the seed
    print(f"\n{'='*62}\n SEED {seed}\n{'='*62}", flush=True)
    for model in MODELS:
        for horizon in HORIZONS:
            if run_cell(seed, model, horizon) != 0:
                failures.append(f"full_{model}_{horizon}min_seed{seed}")

print("\nFailed cells:", failures if failures else "none")

## 7 · Merge the three seeds per cell

Each single-seed run lands in its own parent directory, whose aggregate reports
null for every dispersion field — one seed has no spread. The downstream analysis
expects one parent per cell holding all three seeds, with cross-seed means and
Student-t intervals. This rebuilds that using the benchmark's own aggregation
code, so the merged numbers come from the same code path a three-seed run uses.

Run it once all three seeds of a cell are done; cells that aren't complete are
reported and skipped.

In [ ]:
import subprocess, sys, shutil, pathlib

args = [sys.executable, "merge_seed_runs.py",
        "--seeds", *map(str, SEEDS),
        "--models", *MODELS,
        "--horizons", *map(str, HORIZONS),
        "--copy"]          # copies, not symlinks, so the merged tree stands alone in Drive

print(subprocess.run(args + ["--dry-run"], cwd=REPO_DIR, capture_output=True, text=True).stdout)
result = subprocess.run(args + ["--force"], cwd=REPO_DIR, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)

merged_src = pathlib.Path(REPO_DIR) / "results" / "experiments_merged"
if merged_src.is_dir():
    merged_dst = pathlib.Path(f"{DRIVE_RESULTS}/experiments_merged")
    if merged_dst.exists():
        shutil.rmtree(merged_dst)
    shutil.copytree(merged_src, merged_dst)
    print(f"Merged tree copied to {merged_dst}")

## 8 · Status

In [ ]:
import pathlib, yaml

done = set()
for resolved in EXP_DIR.glob("*/resolved_config.yaml"):
    if (resolved.parent / "aggregate_metrics.json").is_file():
        try:
            done.add(yaml.safe_load(resolved.read_text())["experiment"]["name"])
        except Exception:
            pass

print(f"{'cell':<22}" + "".join(f"seed {s:<6}" for s in SEEDS))
print("-" * (22 + 11 * len(SEEDS)))
complete = 0
for model in MODELS:
    for horizon in HORIZONS:
        cell = f"full_{model}_{horizon}min"
        marks = []
        for s in SEEDS:
            ok = f"{cell}_seed{s}" in done
            complete += ok
            marks.append(f"{'done' if ok else '--':<11}")
        print(f"{cell:<22}" + "".join(marks))
print(f"\n{complete}/{len(MODELS)*len(HORIZONS)*len(SEEDS)} cells complete")
merged = pathlib.Path(REPO_DIR) / "results" / "experiments_merged"
print(f"merged cells: {len(list(merged.glob('full_*'))) if merged.is_dir() else 0}")